# AI-Assisted Code Migration Validation

This notebook explores a modular pipeline for validating AI-generated code migrations.

In [ ]:
# Cell 0 — install dependencies
%pip install -q "openai>=2,<3" "pydantic>=2,<3"

In [ ]:
# Cell 1 — paths and config
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "pipeline":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

SRC_DIR = PROJECT_ROOT / "src"
WORKSPACE_DIR = PROJECT_ROOT / "workspace"

SOURCE_FILE = WORKSPACE_DIR / "source" / "payroll-management.py"
TARGET_FILE = WORKSPACE_DIR / "target" / "payroll-management.cs"

MIGRATION_RULES = """
Preserve the behavior of the original source.
Do not introduce unsupported behavior.
""".strip()

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
# Cell 2 — load source and (optional) target
from common.file_io import read_source_file

source = read_source_file(SOURCE_FILE)
target = read_source_file(TARGET_FILE) if TARGET_FILE.is_file() else None

print(f"Source loaded: {SOURCE_FILE.name}")
print(
    f"Target loaded: {TARGET_FILE.name}"
    if target is not None
    else "Target file is not available yet."
)

In [ ]:
# Cell 3 — configure the LLM
import getpass
from functools import partial

from llm.llm_client import DEFAULT_MODEL, call_llm

api_key = getpass.getpass("Paste your OpenAI API key: ").strip()
if not api_key:
    raise RuntimeError("No OpenAI API key was provided.")

llm_call = partial(call_llm, api_key=api_key, model=DEFAULT_MODEL)

print(f"LLM configured: {DEFAULT_MODEL}")

In [ ]:
# Cell 4 — prompt sanity check (markers consumed, source inserted, JSON intact)
from validation.assets.prompts import (
    EXPECTATION_EXTRACTION_PROMPT,
    RULES_MARKER,
    SOURCE_MARKER,
)

_probe = (
    EXPECTATION_EXTRACTION_PROMPT
    .replace(RULES_MARKER, "")
    .replace(SOURCE_MARKER, "def f(): return 1")
)
assert SOURCE_MARKER not in _probe and RULES_MARKER not in _probe, "markers not consumed"
assert "def f(): return 1" in _probe, "source not inserted"
assert '\"expectations\"' in _probe, "JSON example mangled"
print("Prompt builds correctly.")

In [ ]:
from validation.expectation_extraction import extract_expectations

extraction_result = extract_expectations(
    source=source,
    llm_call=llm_call,
    migration_rules=MIGRATION_RULES,
)

In [ ]:
# Cell 6 — assert the invariants compare_candidate depends on
import re

assert not extraction_result.cannot_validate, extraction_result.reason
for _e in extraction_result.expectations:
    assert re.fullmatch(r"EXP-\d{3}", _e.id), f"bad id: {_e.id}"
print(f"{len(extraction_result.expectations)} expectations, ids well-formed")

In [ ]:
if extraction_result.cannot_validate:
    print("Status: Cannot validate")
    print(f"Reason: {extraction_result.reason}")
else:
    print("Status: Expectations extracted successfully")
    for expectation in extraction_result.expectations:
        print(f"\n{expectation.id}")
        print(f"Category: {expectation.category.value}")
        print(f"Description: {expectation.description}")
        print("Evidence:")
        print(expectation.evidence_quote)

In [ ]:
if target is None:
    raise RuntimeError(
        f"No candidate at {TARGET_FILE}. Place the migrated file there before comparing."
    )

if extraction_result.cannot_validate:
    raise RuntimeError(
        f"Cannot compare: extraction produced no rubric ({extraction_result.reason})."
    )

print(f"Comparing candidate against {len(extraction_result.expectations)} expectations.")

In [ ]:
from validation.compare_candidate import compare_candidate

findings = compare_candidate(
    candidate=target,
    expectations=extraction_result.expectations,
    llm_call=llm_call,
)

print(f"{len(findings)} finding(s) from {len(extraction_result.expectations)} expectations.")

In [ ]:
flagged_ids = {f.expectation_id for f in findings}
preserved = [e for e in extraction_result.expectations if e.id not in flagged_ids]

print(f"Preserved: {len(preserved)} / {len(extraction_result.expectations)}")
for e in preserved:
    print(f"  {e.id}  PRESERVED  {e.description}")

print(f"\nFindings: {len(findings)}")
for f in findings:
    print(f"\n{f.expectation_id}  {f.status.value.upper()}  [{f.severity.value}]")
    print(f"  rationale: {f.rationale}")
    if f.evidence_from_source:
        print(f"  source:    {f.evidence_from_source}")
    if f.evidence_from_candidate:
        print(f"  candidate: {f.evidence_from_candidate}")

In [ ]:
from validation.decide import decide
from common.contracts.verdict import ValidationVerdict

CHECKS_PERFORMED = ("comparison",)   # compile-gate / execution NOT performed

verdict = ValidationVerdict(
    decision=decide(findings),
    source_path=str(SOURCE_FILE),
    candidate_path=str(TARGET_FILE),
    checks_performed=CHECKS_PERFORMED,
    finding_count=len(findings),
)

print(f"Decision:   {verdict.decision.value.upper()}")
print(f"Source:     {verdict.source_path}")
print(f"Candidate:  {verdict.candidate_path}")
print(f"Checks:     {', '.join(verdict.checks_performed)}")
print(f"Findings:   {verdict.finding_count}")
print("\nNote: compilation and execution were not performed. APPROVE means no "
      "blocking issue was found by the checks above, not proven equivalence.")